# 目标检测工程：从框坐标到 IoU、NMS、AP/mAP

目标检测最容易出现“模型没变、指标却变了”的问题：`xywh` 被当成 `xyxy`、resize padding 没有逆映射、NMS 跨类别抑制、相同真值被重复匹配、ignore/crowd 被当普通负例、AP 插值协议不一致。

本 notebook 不调用检测框架的黑盒评估器，而是在受控样例上从零实现坐标变换、IoU、class-aware greedy NMS、Soft-NMS 教学版、one-to-one 匹配、PR/AP/mAP 和版本化后处理，并用反例断言边界。

## 1. 检测系统的外部合同

```text
原图尺寸 / 方向
  -> resize + pad（保存 scale 与 pad）
  -> 模型输出 boxes/logits
  -> 解码到模型输入坐标
  -> clip / 去退化框 / 置信阈值
  -> 逆映射到原图
  -> class-aware NMS 或 Soft-NMS
  -> 固定协议的 one-to-one matching
  -> PR、AP、mAP 与延迟报告
```

每个阶段都应写入 trace：坐标约定、模型输入尺寸、阈值、NMS 类型、IoU 阈值和代码版本。最终框没有这些元数据，就很难解释线上与离线差异。

In [ ]:
from dataclasses import dataclass
from collections import Counter, defaultdict
import time
import numpy as np

# 坐标统一为原图像素、左上角原点、半开区间 [x1,y1,x2,y2)。
GROUND_TRUTH = [
    {"image_id": "img-1", "class_id": 0, "box": [10, 10, 50, 50], "ignore": False, "crowd": False},
    {"image_id": "img-1", "class_id": 1, "box": [60, 10, 95, 45], "ignore": False, "crowd": False},
    {"image_id": "img-1", "class_id": 0, "box": [0, 0, 8, 8], "ignore": True, "crowd": False},
    {"image_id": "img-2", "class_id": 0, "box": [20, 20, 60, 70], "ignore": False, "crowd": False},
    {"image_id": "img-2", "class_id": 1, "box": [65, 25, 95, 65], "ignore": False, "crowd": True},
]
PREDICTIONS = [
    {"image_id": "img-1", "class_id": 0, "score": .95, "box": [11, 11, 49, 50]},
    {"image_id": "img-1", "class_id": 0, "score": .80, "box": [12, 12, 48, 49]},  # duplicate
    {"image_id": "img-1", "class_id": 1, "score": .90, "box": [62, 12, 94, 44]},
    {"image_id": "img-1", "class_id": 1, "score": .30, "box": [0, 70, 20, 90]},   # background
    {"image_id": "img-1", "class_id": 0, "score": .40, "box": [0, 0, 7, 7]},      # ignore overlap
    {"image_id": "img-2", "class_id": 0, "score": .88, "box": [19, 22, 61, 69]},
    {"image_id": "img-2", "class_id": 0, "score": .65, "box": [65, 25, 95, 65]}, # wrong class
    {"image_id": "img-2", "class_id": 1, "score": .72, "box": [66, 26, 94, 64]}, # crowd
    {"image_id": "img-2", "class_id": 1, "score": .60, "box": [67, 27, 93, 63]}, # crowd again
]
IMAGE_SHAPES = {"img-1": (100, 100), "img-2": (100, 100)}  # (height,width)
print("GT:", len(GROUND_TRUTH), "predictions:", len(PREDICTIONS))


## 2. 框坐标、clip 和 resize 逆映射

本例选择浮点 `xyxy=[x1,y1,x2,y2]` 半开区间，因此宽高为 `x2-x1,y2-y1`；另一套库可能使用闭区间并出现 `+1`。协议不一致会系统性改变小目标 IoU。

letterbox resize 不能只保存缩放后图片：必须保存 `scale、pad_x、pad_y、原图尺寸、目标尺寸`。逆映射先减 padding 再除 scale，最后 clip 到原图。退化框（宽或高为 0）保留为可诊断输入，但面积和 IoU 都是 0；负宽高则直接拒绝。

In [ ]:
def as_boxes(boxes):
    array = np.asarray(boxes, dtype=np.float64)
    if array.size == 0:
        return np.empty((0, 4), dtype=np.float64)
    if array.ndim == 1:
        if array.shape != (4,):
            raise ValueError("单个 box 必须有四个坐标")
        array = array.reshape(1, 4)
    if array.ndim != 2 or array.shape[1] != 4 or not np.isfinite(array).all():
        raise ValueError("boxes 必须是有限的 [N,4]")
    if np.any(array[:, 2] < array[:, 0]) or np.any(array[:, 3] < array[:, 1]):
        raise ValueError("xyxy 不能出现负宽或负高")
    return array

def validate_image_hw(image_hw, name="image_hw"):
    array = np.asarray(image_hw, dtype=float)
    if array.shape != (2,) or not np.isfinite(array).all() or np.any(array <= 0):
        raise ValueError(f"{name} 必须是两个有限正数 (height,width)")
    return float(array[0]), float(array[1])

def validate_probability_threshold(value, name):
    if not np.isfinite(value) or not 0 <= value <= 1:
        raise ValueError(f"{name} 必须是 [0,1] 有限数")

def validate_detection_arrays(boxes, scores, class_ids):
    boxes = as_boxes(boxes)
    scores = np.asarray(scores, dtype=float)
    raw_classes = np.asarray(class_ids)
    if scores.ndim != 1 or raw_classes.ndim != 1:
        raise ValueError("scores/class_ids 必须是一维")
    if len(boxes) != len(scores) or len(boxes) != len(raw_classes):
        raise ValueError("boxes/scores/class_ids 长度不一致")
    if not np.isfinite(scores).all() or np.any((scores < 0) | (scores > 1)):
        raise ValueError("scores 必须是 [0,1] 有限概率")
    try:
        numeric_classes = raw_classes.astype(float)
    except (TypeError, ValueError):
        raise ValueError("class_ids 必须是有限整数") from None
    if not np.isfinite(numeric_classes).all() or not np.all(numeric_classes == np.floor(numeric_classes)):
        raise ValueError("class_ids 必须是有限整数")
    return boxes, scores, numeric_classes.astype(np.int64)

def xywh_to_xyxy(boxes):
    array = np.asarray(boxes, dtype=np.float64)
    if array.size == 0:
        return np.empty((0, 4), dtype=np.float64)
    if array.ndim == 1:
        if array.shape != (4,):
            raise ValueError("单个 xywh 必须有四个值")
        array = array.reshape(1, 4)
    if array.ndim != 2 or array.shape[1] != 4 or not np.isfinite(array).all():
        raise ValueError("xywh 必须是有限 [N,4]")
    if np.any(array[:, 2:] < 0):
        raise ValueError("xywh 宽高必须非负")
    out = array.copy()
    out[:, 2] = array[:, 0] + array[:, 2]
    out[:, 3] = array[:, 1] + array[:, 3]
    return out

def xyxy_to_xywh(boxes):
    boxes = as_boxes(boxes)
    out = boxes.copy()
    out[:, 2] = boxes[:, 2] - boxes[:, 0]
    out[:, 3] = boxes[:, 3] - boxes[:, 1]
    return out

def clip_boxes(boxes, image_hw):
    boxes = as_boxes(boxes).copy()
    height, width = validate_image_hw(image_hw)
    boxes[:, [0, 2]] = np.clip(boxes[:, [0, 2]], 0, width)
    boxes[:, [1, 3]] = np.clip(boxes[:, [1, 3]], 0, height)
    return boxes

def letterbox_map(boxes, source_hw, target_hw):
    source_h, source_w = validate_image_hw(source_hw, "source_hw")
    target_h, target_w = validate_image_hw(target_hw, "target_hw")
    scale = min(target_w / source_w, target_h / source_h)
    pad_x = (target_w - source_w * scale) / 2.0
    pad_y = (target_h - source_h * scale) / 2.0
    mapped = as_boxes(boxes).copy() * scale
    mapped[:, [0, 2]] += pad_x
    mapped[:, [1, 3]] += pad_y
    meta = {"scale": scale, "pad_x": pad_x, "pad_y": pad_y,
            "source_hw": (source_h, source_w), "target_hw": (target_h, target_w)}
    return mapped, meta

def letterbox_inverse(boxes, meta):
    required = {"scale", "pad_x", "pad_y", "source_hw"}
    if not required.issubset(meta) or not np.isfinite([meta["scale"], meta["pad_x"], meta["pad_y"]]).all():
        raise ValueError("letterbox meta 缺失或含非有限值")
    if meta["scale"] <= 0:
        raise ValueError("letterbox scale 必须为正")
    restored = as_boxes(boxes).copy()
    restored[:, [0, 2]] = (restored[:, [0, 2]] - meta["pad_x"]) / meta["scale"]
    restored[:, [1, 3]] = (restored[:, [1, 3]] - meta["pad_y"]) / meta["scale"]
    return clip_boxes(restored, meta["source_hw"])

probe = np.array([[10, 5, 90, 45]], dtype=float)
mapped, resize_meta = letterbox_map(probe, (50, 100), (128, 128))
restored = letterbox_inverse(mapped, resize_meta)
assert np.allclose(restored, probe)
assert np.allclose(xywh_to_xyxy(xyxy_to_xywh(probe)), probe)
print("mapped:", mapped.round(2).tolist(), "meta:", resize_meta)


## 3. IoU：几何合同的第一道单测

$IoU(A,B)=|A\cap B|/|A\cup B|$。实现应向量化，并明确空并集、接触边界、退化框的返回值。本例把它们定义为 0。对旋转框、mask、3D box 则需要另一套几何与协议，不能复用轴对齐框函数冒充。

In [ ]:
def box_area(boxes):
    boxes = as_boxes(boxes)
    return np.maximum(0.0, boxes[:, 2] - boxes[:, 0]) * np.maximum(0.0, boxes[:, 3] - boxes[:, 1])

def box_iou(boxes_a, boxes_b):
    a, b = as_boxes(boxes_a), as_boxes(boxes_b)
    top_left = np.maximum(a[:, None, :2], b[None, :, :2])
    bottom_right = np.minimum(a[:, None, 2:], b[None, :, 2:])
    wh = np.maximum(0.0, bottom_right - top_left)
    intersection = wh[..., 0] * wh[..., 1]
    union = box_area(a)[:, None] + box_area(b)[None, :] - intersection
    return np.divide(intersection, union, out=np.zeros_like(intersection), where=union > 0)

assert np.isclose(box_iou([[0, 0, 10, 10]], [[0, 0, 10, 10]])[0, 0], 1.0)
assert box_iou([[0, 0, 10, 10]], [[10, 0, 20, 10]])[0, 0] == 0.0
assert box_iou([[1, 1, 1, 5]], [[1, 1, 1, 5]])[0, 0] == 0.0
print(box_iou([[0, 0, 10, 10], [0, 0, 5, 5]], [[0, 0, 10, 10]]).ravel())


## 4. Greedy NMS：排序稳定性与类别边界

标准 greedy NMS 按置信度从高到低选择框，再删除与它 IoU 超阈值的候选。相同分数必须有稳定 tie-break，否则不同硬件/并行顺序可能产生不同输出。

默认应该 class-aware：猫框不应抑制同位置的狗框。若业务确实要求 class-agnostic，需要作为显式、版本化配置。NMS 阈值与 score 阈值必须在 validation 上联合调节。

In [ ]:
def greedy_class_aware_nms(boxes, scores, class_ids, iou_threshold=0.5):
    boxes, scores, class_ids = validate_detection_arrays(boxes, scores, class_ids)
    validate_probability_threshold(iou_threshold, "iou_threshold")
    keep = []
    for class_id in np.unique(class_ids):
        indices = np.flatnonzero(class_ids == class_id)
        order = indices[np.lexsort((indices, -scores[indices]))]
        while len(order):
            current = int(order[0])
            keep.append(current)
            if len(order) == 1:
                break
            overlaps = box_iou(boxes[[current]], boxes[order[1:]])[0]
            order = order[1:][overlaps <= iou_threshold]
    return np.array(sorted(keep, key=lambda i: (-scores[i], i)), dtype=int)

probe_boxes = np.array([[0, 0, 10, 10], [1, 1, 9, 9], [0, 0, 10, 10]], float)
probe_scores = np.array([.9, .8, .85])
probe_classes = np.array([0, 0, 1])
kept = greedy_class_aware_nms(probe_boxes, probe_scores, probe_classes, .5)
assert kept.tolist() == [0, 2]
print("kept indices:", kept.tolist())


## 5. Soft-NMS：衰减而不是删除

拥挤场景中，硬删除可能把邻近真实目标一起移除。Soft-NMS 根据 IoU 衰减分数，再重新排序。下面实现线性衰减的教学版本；原论文还讨论 Gaussian 形式。

注意：衰减后 score 的分布改变，旧置信阈值和校准不再有效。生产切换 NMS 算法时，应连同阈值、评估协议和后处理版本一起发布。

In [ ]:
def linear_soft_nms(boxes, scores, class_ids, iou_threshold=0.5, min_score=0.05):
    boxes, adjusted, class_ids = validate_detection_arrays(boxes, scores, class_ids)
    adjusted = adjusted.copy()
    validate_probability_threshold(iou_threshold, "iou_threshold")
    validate_probability_threshold(min_score, "min_score")
    selected = []
    for class_id in np.unique(class_ids):
        remaining = list(np.flatnonzero(class_ids == class_id))
        while remaining:
            current = min(remaining, key=lambda i: (-adjusted[i], i))
            remaining.remove(current)
            if adjusted[current] < min_score:
                break
            selected.append((int(current), float(adjusted[current])))
            if remaining:
                overlaps = box_iou(boxes[[current]], boxes[remaining])[0]
                for index, overlap in zip(remaining, overlaps):
                    if overlap > iou_threshold:
                        adjusted[index] *= 1.0 - overlap
                remaining = [index for index in remaining if adjusted[index] >= min_score]
    return sorted(selected, key=lambda item: (-item[1], item[0])), adjusted

soft_kept, soft_scores = linear_soft_nms(probe_boxes, probe_scores, probe_classes, .5, .01)
assert soft_scores[1] < probe_scores[1]
assert soft_scores[2] == probe_scores[2]
print(soft_kept)


## 6. 一对一匹配：高分预测先占用真值

对某个类别和 IoU 阈值，预测按 score 全局降序。每个普通 GT 最多匹配一次：首个达到阈值的预测记 TP，之后覆盖同一 GT 的预测记 duplicate FP。没有普通匹配、但覆盖 ignore/crowd 区域的预测不进入 TP/FP。

下面对 crowd 使用“可吸收多个预测”的简化规则，用来展示边界；它**不是 COCO 官方评估器的完整复刻**，没有 area range、maxDets、mask crowd IoU 等细节。正式 COCO 报告应使用固定版本 `pycocotools` 并记录参数。

In [ ]:
def crowd_overlap(det_boxes, crowd_boxes):
    """COCO crowd criterion: intersection(det,crowd) / area(det)."""
    detections, crowds = as_boxes(det_boxes), as_boxes(crowd_boxes)
    top_left = np.maximum(detections[:, None, :2], crowds[None, :, :2])
    bottom_right = np.minimum(detections[:, None, 2:], crowds[None, :, 2:])
    wh = np.maximum(0.0, bottom_right - top_left)
    intersection = wh[..., 0] * wh[..., 1]
    det_area = box_area(detections)[:, None]
    return np.divide(intersection, det_area, out=np.zeros_like(intersection), where=det_area > 0)

def match_image_class(predictions, ground_truth, iou_threshold):
    validate_probability_threshold(iou_threshold, "iou_threshold")
    if predictions:
        validate_detection_arrays([p["box"] for p in predictions],
                                  [p["score"] for p in predictions],
                                  [0] * len(predictions))
    if ground_truth:
        as_boxes([g["box"] for g in ground_truth])
    predictions = sorted(enumerate(predictions), key=lambda pair: (-pair[1]["score"], pair[0]))
    regular = [g for g in ground_truth if not g.get("ignore", False) and not g.get("crowd", False)]
    ignored_regular = [g for g in ground_truth if g.get("ignore", False) and not g.get("crowd", False)]
    crowds = [g for g in ground_truth if g.get("crowd", False)]
    used = set()
    results = []
    for original_order, prediction in predictions:
        pbox = [prediction["box"]]
        regular_iou = box_iou(pbox, [g["box"] for g in regular])[0] if regular else np.array([])
        candidates = [i for i, value in enumerate(regular_iou)
                      if value >= iou_threshold and i not in used]
        duplicate = any(value >= iou_threshold and i in used for i, value in enumerate(regular_iou))
        if candidates:
            best = max(candidates, key=lambda i: (regular_iou[i], -i))
            used.add(best)
            status, matched, reason = "TP", best, "regular_match"
        else:
            ignored_iou = (box_iou(pbox, [g["box"] for g in ignored_regular])[0]
                           if ignored_regular else np.array([]))
            crowd_iou = (crowd_overlap(pbox, [g["box"] for g in crowds])[0]
                         if crowds else np.array([]))
            if len(ignored_iou) and ignored_iou.max() >= iou_threshold:
                status, matched, reason = "IGNORED", None, "ignore_region"
            elif len(crowd_iou) and crowd_iou.max() >= iou_threshold:
                status, matched, reason = "IGNORED", None, "crowd_region"
            else:
                status, matched, reason = "FP", None, "duplicate" if duplicate else "unmatched"
        results.append({"score": float(prediction["score"]), "status": status,
                        "reason": reason, "order": original_order,
                        "matched_regular": matched})
    return results, len(regular)

img1_cls0_pred = [p for p in PREDICTIONS if p["image_id"] == "img-1" and p["class_id"] == 0]
img1_cls0_gt = [g for g in GROUND_TRUTH if g["image_id"] == "img-1" and g["class_id"] == 0]
match_demo, positive_count = match_image_class(img1_cls0_pred, img1_cls0_gt, .5)
assert [row["status"] for row in match_demo] == ["TP", "FP", "IGNORED"]
assert match_demo[1]["reason"] == "duplicate"
assert positive_count == 1
CONTAINED_CROWD = match_image_class(
    [{"score": .9, "box": [0, 0, 10, 10]}],
    [{"box": [0, 0, 100, 100], "ignore": False, "crowd": True}], .5
)[0]
assert CONTAINED_CROWD[0]["status"] == "IGNORED"
print(match_demo)


## 7. Precision–Recall 与 AP 协议

按 score 阈值从高到低扫描：`precision=累计TP/(累计TP+累计FP)`，`recall=累计TP/普通GT数`。AP 是 precision–recall 曲线面积，但不同协议并不相同：VOC 2007 曾用 11 点插值，后续 VOC 使用全点积分；COCO 使用 101 个 recall 点，并对多个 IoU、类别、面积和 maxDets 聚合。

因此“mAP=0.42”缺少协议就没有可比性。下面同时实现全点积分与 101 点插值。

In [ ]:
def precision_recall(matched_rows, positives):
    active = [row for row in matched_rows if row["status"] != "IGNORED"]
    active = sorted(enumerate(active), key=lambda pair: (-pair[1]["score"], pair[0]))
    tp = np.array([row["status"] == "TP" for _, row in active], dtype=float)
    fp = 1.0 - tp
    cum_tp, cum_fp = np.cumsum(tp), np.cumsum(fp)
    precision = np.divide(cum_tp, cum_tp + cum_fp, out=np.zeros_like(cum_tp), where=(cum_tp + cum_fp) > 0)
    recall = cum_tp / positives if positives else np.zeros_like(cum_tp)
    scores = np.array([row["score"] for _, row in active], dtype=float)
    return precision, recall, scores

def interpolated_ap(precision, recall, points=None):
    precision, recall = np.asarray(precision), np.asarray(recall)
    if len(precision) == 0:
        return 0.0
    if points is not None:
        grid = np.linspace(0, 1, points)
        values = [precision[recall >= level].max() if np.any(recall >= level) else 0.0 for level in grid]
        return float(np.mean(values))
    mrec = np.concatenate([[0.0], recall, [1.0]])
    mpre = np.concatenate([[0.0], precision, [0.0]])
    for index in range(len(mpre) - 2, -1, -1):
        mpre[index] = max(mpre[index], mpre[index + 1])
    changes = np.flatnonzero(mrec[1:] != mrec[:-1])
    return float(np.sum((mrec[changes + 1] - mrec[changes]) * mpre[changes + 1]))

precision_demo, recall_demo, _ = precision_recall(match_demo, positive_count)
assert np.isclose(interpolated_ap(precision_demo, recall_demo), 1.0)
assert np.isclose(interpolated_ap(precision_demo, recall_demo, points=101), 1.0)
print("precision:", precision_demo, "recall:", recall_demo)


## 8. 从单图匹配到数据集 mAP

匹配在每个 `image_id × class_id` 内完成，随后该类别的预测跨图片按 score 汇总。分母是该类别所有非 ignore/crowd 真值。最后先对类别求平均，再对 IoU 阈值求平均。

类别没有 GT 时应排除还是计 0，必须由协议规定。本例只评估 fixture 中有普通 GT 的类别。正式评估还需要空预测、空 GT、重复 image id、未知 class id 和最大检测数的契约。

In [ ]:
def evaluate_class(predictions, ground_truth, class_id, iou_threshold):
    image_ids = sorted({row["image_id"] for row in predictions + ground_truth})
    matched, positives = [], 0
    for image_id in image_ids:
        pred = [p for p in predictions if p["image_id"] == image_id and p["class_id"] == class_id]
        gt = [g for g in ground_truth if g["image_id"] == image_id and g["class_id"] == class_id]
        image_rows, image_positives = match_image_class(pred, gt, iou_threshold)
        matched.extend(image_rows)
        positives += image_positives
    precision, recall, scores = precision_recall(matched, positives)
    return {"ap_all_points": interpolated_ap(precision, recall),
            "ap_101": interpolated_ap(precision, recall, 101),
            "precision": precision, "recall": recall, "scores": scores,
            "positives": positives, "matched": matched}

def evaluate_map(predictions, ground_truth, classes=(0, 1), thresholds=(.5, .75)):
    details = {}
    for threshold in thresholds:
        for class_id in classes:
            details[(threshold, class_id)] = evaluate_class(
                predictions, ground_truth, class_id, threshold
            )
    mean_ap = float(np.mean([row["ap_101"] for row in details.values()]))
    return mean_ap, details

mean_ap, map_details = evaluate_map(PREDICTIONS, GROUND_TRUTH)
print("teaching mAP@[.50,.75]:", round(mean_ap, 4),
      {str(key): round(value["ap_101"], 3) for key, value in map_details.items()})
assert 0.0 <= mean_ap <= 1.0
assert map_details[(.5, 0)]["positives"] == 2
assert map_details[(.5, 1)]["positives"] == 1


## 9. 置信阈值、NMS 阈值与延迟要一起看

降低 score 阈值提高候选召回，也增加 NMS 与下游存储成本；降低 NMS IoU 阈值减少重复框，也可能压掉密集目标。AP 通常需要保留较低分候选形成完整 PR 曲线，不能先用线上高阈值截断后再声称官方 AP。

下面的 sweep 只是证明参数如何影响候选数与教学 mAP。微秒级计时受机器噪声影响，只能验证计时代码路径；生产需在真实 batch、输入尺寸、硬件和并发下报告 p50/p95/p99。

In [ ]:
def postprocess_fixture(predictions, score_threshold, nms_iou):
    output = []
    for image_id in sorted({p["image_id"] for p in predictions}):
        rows = [p for p in predictions if p["image_id"] == image_id and p["score"] >= score_threshold]
        if not rows:
            continue
        keep = greedy_class_aware_nms(
            [r["box"] for r in rows], [r["score"] for r in rows],
            [r["class_id"] for r in rows], nms_iou
        )
        output.extend(rows[i] for i in keep)
    return output

sweep = []
for score_threshold in (.05, .50, .75):
    for nms_iou in (.3, .5, .7):
        start = time.perf_counter()
        post = postprocess_fixture(PREDICTIONS, score_threshold, nms_iou)
        elapsed_us = (time.perf_counter() - start) * 1e6
        score, _ = evaluate_map(post, GROUND_TRUTH)
        sweep.append((score_threshold, nms_iou, len(post), score, elapsed_us))
print("score_thr nms_iou count teaching_mAP elapsed_us")
for row in sweep:
    print(tuple(round(value, 4) if isinstance(value, float) else value for value in row))
assert all(row[2] <= len(PREDICTIONS) for row in sweep)


## 10. AP 下降要拆成可行动错误

常见类别：`background`（不覆盖任何 GT）、`localization`（类别对但 IoU 不足）、`classification`（位置对但类别错）、`duplicate`（普通 GT 已被更高分预测占用）、`miss`（没有任何预测覆盖 GT）。

诊断时比较多个 IoU：在 0.5 是 TP、0.75 变 FP，通常是定位；在所有阈值都失败可能是分类或召回。还要按目标面积、遮挡、场景、设备和时间分群，而非只看一张 PR 曲线。

In [ ]:
def detection_error_analysis(predictions, ground_truth, iou_threshold=.5):
    """先复用 matching，再只对 unmatched FP 做跨类/定位诊断。"""
    validate_probability_threshold(iou_threshold, "iou_threshold")
    records, matched_gt = [], set()
    groups = sorted({(r["image_id"], r["class_id"]) for r in predictions + ground_truth})
    for image_id, class_id in groups:
        indexed = [(i, p) for i, p in enumerate(predictions)
                   if p["image_id"] == image_id and p["class_id"] == class_id]
        class_gt = [g for g in ground_truth
                    if g["image_id"] == image_id and g["class_id"] == class_id]
        regular = [g for g in class_gt if not g.get("ignore", False) and not g.get("crowd", False)]
        matched_rows, _ = match_image_class([p for _, p in indexed], class_gt, iou_threshold)
        for row in matched_rows:
            global_index, prediction = indexed[row["order"]]
            if row["status"] == "TP":
                matched_gt.add((image_id, class_id, row["matched_regular"]))
                kind = "tp"
            elif row["status"] == "IGNORED":
                kind = "ignored"
            elif row["reason"] == "duplicate":
                kind = "duplicate"
            else:
                candidates = [g for g in ground_truth if g["image_id"] == image_id
                              and not g.get("ignore", False) and not g.get("crowd", False)]
                if not candidates:
                    kind = "background"
                else:
                    overlaps = box_iou([prediction["box"]], [g["box"] for g in candidates])[0]
                    best = int(np.argmax(overlaps))
                    target = candidates[best]
                    if overlaps[best] < .1:
                        kind = "background"
                    elif target["class_id"] != class_id and overlaps[best] >= iou_threshold:
                        kind = "classification"
                    elif target["class_id"] == class_id and overlaps[best] < iou_threshold:
                        kind = "localization"
                    else:
                        kind = "background"
            records.append({"type": kind, "prediction_index": global_index,
                            "image_id": image_id, "class_id": class_id,
                            "match_status": row["status"], "match_reason": row["reason"]})
        for local_index, gt in enumerate(regular):
            if (image_id, class_id, local_index) not in matched_gt:
                records.append({"type": "miss", "prediction_index": None,
                                "image_id": image_id, "class_id": class_id, "gt_box": gt["box"]})
    return records

ERROR_RECORDS = detection_error_analysis(PREDICTIONS, GROUND_TRUTH)
error_taxonomy = Counter(row["type"] for row in ERROR_RECORDS)
CLASSIFICATION_PROBE = detection_error_analysis(
    [{"image_id": "img-1", "class_id": 1, "score": .2, "box": [10, 10, 50, 50]}],
    GROUND_TRUTH, .5
)
print(error_taxonomy)
assert error_taxonomy["duplicate"] >= 1
assert error_taxonomy["ignored"] >= 1
assert error_taxonomy["background"] >= 1
assert any(row["type"] == "classification" for row in CLASSIFICATION_PROBE)
assert all(not (row["type"] in {"classification", "localization"} and row["match_status"] == "IGNORED")
           for row in ERROR_RECORDS if "match_status" in row)


## 11. 把后处理配置变成版本化组件

模型导出的 raw boxes 不是稳定业务结果。score threshold、坐标解码、clip、最小面积、NMS 类型和阈值都会改变输出，应与模型形成不可分割的 bundle。返回值还应包含原图尺寸和变换 trace，避免调用方二次误缩放。

In [ ]:
@dataclass(frozen=True)
class DetectionPostprocessor:
    score_threshold: float = .25
    nms_iou: float = .50
    min_area: float = 1.0
    max_detections: int = 100
    version: str = "xyxy-halfopen-classnms-v2"

    def __post_init__(self):
        validate_probability_threshold(self.score_threshold, "score_threshold")
        validate_probability_threshold(self.nms_iou, "nms_iou")
        if not np.isfinite(self.min_area) or self.min_area < 0:
            raise ValueError("min_area 必须是有限非负数")
        if isinstance(self.max_detections, bool) or not isinstance(self.max_detections, (int, np.integer)) or self.max_detections <= 0:
            raise ValueError("max_detections 必须是正整数")
        if not self.version:
            raise ValueError("version 不能为空")

    def __call__(self, boxes, scores, class_ids, image_hw):
        validate_image_hw(image_hw)
        boxes, scores, class_ids = validate_detection_arrays(boxes, scores, class_ids)
        boxes = clip_boxes(boxes, image_hw)
        valid = (box_area(boxes) >= self.min_area) & (scores >= self.score_threshold)
        original = np.flatnonzero(valid)
        if not len(original):
            return []
        relative = greedy_class_aware_nms(boxes[valid], scores[valid], class_ids[valid], self.nms_iou)
        keep = original[relative][:self.max_detections]
        return [{"box": boxes[i].tolist(), "score": float(scores[i]),
                 "class_id": int(class_ids[i]), "postprocess_version": self.version}
                for i in keep]

postprocessor = DetectionPostprocessor()
served = postprocessor(probe_boxes, probe_scores, probe_classes, (10, 10))
assert len(served) == 2
assert all(row["postprocess_version"] == postprocessor.version for row in served)
print(served)


## 12. 边界与回归测试

检测测试不能只放一组正常框。最低集合包括：完全重合、不相交、仅接边、包含关系、退化框、负坐标、越界、resize 往返、同分排序、跨类别重合、重复检测、ignore/crowd、多图片全局排序和空输出。

In [ ]:
assert np.allclose(xywh_to_xyxy([[2, 3, 5, 7]]), [[2, 3, 7, 10]])
assert np.allclose(xyxy_to_xywh([[2, 3, 7, 10]]), [[2, 3, 5, 7]])
assert np.allclose(clip_boxes([[-2, -3, 12, 11]], (10, 10)), [[0, 0, 10, 10]])
assert np.allclose(letterbox_inverse(*letterbox_map([[3, 4, 30, 40]], (50, 80), (640, 640))), [[3, 4, 30, 40]])
assert box_area([[0, 0, 0, 5]])[0] == 0
assert np.isclose(box_iou([[0, 0, 10, 10]], [[2, 2, 8, 8]])[0, 0], .36)
assert np.allclose(box_iou([[0, 0, 5, 5]], [[0, 0, 5, 5], [5, 0, 10, 5]]), [[1, 0]])
assert greedy_class_aware_nms(probe_boxes, probe_scores, probe_classes, .5).tolist() == [0, 2]
tie_keep = greedy_class_aware_nms([[0, 0, 10, 10], [1, 1, 9, 9]], [.8, .8], [0, 0], .5)
assert tie_keep.tolist() == [0]
assert soft_scores[1] < .8 and soft_scores[1] >= 0
assert [row["status"] for row in match_demo].count("TP") == 1
assert [row["reason"] for row in match_demo].count("duplicate") == 1
crowd_pred = [p for p in PREDICTIONS if p["image_id"] == "img-2" and p["class_id"] == 1]
crowd_gt = [g for g in GROUND_TRUTH if g["image_id"] == "img-2" and g["class_id"] == 1]
crowd_match, crowd_positive = match_image_class(crowd_pred, crowd_gt, .5)
assert crowd_positive == 0
assert all(row["status"] == "IGNORED" and row["reason"] == "crowd_region" for row in crowd_match)
assert crowd_overlap([[0, 0, 10, 10]], [[0, 0, 100, 100]])[0, 0] == 1.0
assert CONTAINED_CROWD[0]["status"] == "IGNORED"
assert np.all(np.diff(map_details[(.5, 0)]["recall"]) >= 0)
assert 0 <= interpolated_ap([1, .5], [.5, 1.0]) <= 1
assert len(postprocessor([], [], [], (10, 10))) == 0
assert error_taxonomy["duplicate"] >= 1 and error_taxonomy["ignored"] >= 1
assert any(row["type"] == "classification" for row in CLASSIFICATION_PROBE)

for bad_call in [
    lambda: as_boxes([[5, 5, 4, 8]]),
    lambda: xywh_to_xyxy([[np.nan, 0, 1, 1]]),
    lambda: clip_boxes([[0, 0, 1, 1]], (0, 10)),
    lambda: letterbox_map([[0, 0, 1, 1]], (10, 0), (10, 10)),
    lambda: greedy_class_aware_nms(probe_boxes, probe_scores[:2], probe_classes, .5),
    lambda: greedy_class_aware_nms(probe_boxes, [np.nan, .8, .7], probe_classes, .5),
    lambda: linear_soft_nms(probe_boxes, probe_scores, probe_classes[:2]),
    lambda: linear_soft_nms(probe_boxes, probe_scores, probe_classes, iou_threshold=1.5),
    lambda: DetectionPostprocessor(nms_iou=1.5),
    lambda: DetectionPostprocessor(min_area=-1),
    lambda: DetectionPostprocessor(max_detections=0),
]:
    try:
        bad_call()
        raise AssertionError("非法检测输入应被拒绝")
    except ValueError:
        pass
print("目标检测契约测试通过：几何、crowd、匹配、指标与非法输入均已覆盖")


## 13. 生产边界与原始资料

**不要混淆的边界**：本例是轴对齐框；不是 mask/旋转框。crowd 逻辑是教学简化；不是 COCO evaluator 等价实现。小型 fixture 能证明数学与契约；不能预测真实模型质量。微基准能检查路径；不能代表服务尾延迟。

**上线清单**：固定坐标与 resize 版本；黄金图往返误差；原始输出和最终输出可追踪；score/NMS/最大检测数联合回归；官方 evaluator 容器与版本锁定；按类别/面积/场景误差分析；空图、超大图、NaN、未知类保护；真实并发 p50/p95/p99；灰度与回滚。

**资料**：

- Everingham et al., *The PASCAL Visual Object Classes Challenge*, IJCV 2010：https://doi.org/10.1007/s11263-009-0275-4
- Lin et al., *Microsoft COCO: Common Objects in Context*, ECCV 2014：https://arxiv.org/abs/1405.0312
- COCO 官方 API / evaluator：https://github.com/cocodataset/cocoapi
- Neubeck & Van Gool, *Efficient Non-Maximum Suppression*, ICPR 2006：https://doi.org/10.1109/ICPR.2006.479
- Bodla et al., *Soft-NMS — Improving Object Detection With One Line of Code*, ICCV 2017：https://arxiv.org/abs/1704.04503